# SegRNN — one-notebook reproduction (reconstruction + improvement)

Runtime → Change runtime type → **T4 GPU** before running. Dataset CSVs must
already be in Google Drive at `MyDrive/ts-project/dataset/`.

**Run every cell top to bottom, once.** Nothing needs to be copied out and
pasted back anywhere — every table and graph is produced and shown inline,
right here.

What this notebook does, in order:
- **Part 1** — trains and evaluates the original **SegRNN** (the paper
  reconstruction) on ETTh1, all four horizons.
- **Part 2** — computes the classical baselines (naive, seasonal-naive) on
  the identical data pipeline, deterministic, no GPU needed.
- **Part 3** — table + graphs: paper vs. reconstruction vs. baselines.
- **Part 4** — trains and evaluates **SegRNNTime**, the Stage 2 improved
  architecture (calendar features wired into the decoder's positional
  embedding — see `docs/stage2_report_draft.md` for why the encoder was
  abandoned).
- **Part 5** — final table + graphs (new figures): paper vs. reconstruction
  vs. baselines vs. improved.
- **Part 6** — seed variance check: reruns Reconstruction and Improved at
  2 more seeds for the two most relevant horizons, to test whether the
  small deltas in Part 5 are a real effect or seed noise.
- **Part 7** — efficiency sweep: reruns the reconstruction at
  `d_model ∈ {512, 256, 128, 64}` across all four horizons — a second,
  independent Stage 2 improvement candidate using 100% unmodified code,
  tracing an accuracy-vs-inference-cost curve.
- **Part 8** — RevIN normalization: reruns the reconstruction with
  `--revin 1` — a third improvement candidate, already implemented in the
  repo and just switched on.
- **Part 9** — attention over encoder states (`SegRNNAttn`): a fourth
  candidate, and the most structural one — removes the encoder's
  single-hidden-state bottleneck instead of adding more information for
  it to carry.
- **Optional** — save results/figures back into the repo and push.

Expect roughly 1.5–2 hours total on a T4 (Parts 1, 4, 7, 8, 9 each train
multiple models with early stopping — Part 7 alone is 16 runs; Part 6
trains 8 more; Parts 2/3/5 take seconds).

The actual method code lives in the repo as usual —
[`models/SegRNN.py`](../models/SegRNN.py),
[`models/SegRNNTime.py`](../models/SegRNNTime.py),
[`models/SegRNNAttn.py`](../models/SegRNNAttn.py),
[`run_longExp.py`](../run_longExp.py),
[`scripts/baselines.py`](../scripts/baselines.py) — this notebook just
calls it and organizes the output into one place.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

Run this once. `run_horizon` launches `run_longExp.py` as a subprocess (the
exact same entry point the paper's own scripts use), streams its output live
so you can watch training progress, and parses the final `mse:.., mae:..`
line it prints — no manual copying, no stale results.

In [ ]:
import os, sys, re, csv, subprocess, datetime, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]

# Paper's Table II, ETTh1, multivariate, L=720 (docs/SegRNN_paper.pdf)
PAPER = {
    'mse': {96: 0.351, 192: 0.392, 336: 0.423, 720: 0.466},
    'mae': {96: 0.392, 192: 0.414, 336: 0.433, 720: 0.472},
}

# Reconstruction baseline (SegRNN, d_model=512, seed=2024), from results/runs.csv --
# same comparison point Parts 1/4/5/8/9 all use. Hardcoded (like PAPER above) so
# Parts 0a-0d below don't require re-running Part 1 first.
RECON_BASELINE = {
    'mse': {96: 0.3510, 192: 0.3925, 336: 0.4233, 720: 0.4657},
    'mae': {96: 0.3925, 192: 0.4142, 336: 0.4327, 720: 0.4719},
}

os.makedirs('results/figures', exist_ok=True)


def run_horizon(model, pred_len, mark_dim=None, hour_emb_dim=None, weekday_emb_dim=None, seed=None,
                 d_model=512, revin=None, power_transform=None, save_preds=None,
                 rocket_kernels=None, rocket_kernel_size=None):
    """Launch run_longExp.py for one (model, horizon), stream its output
    live, and parse the final 'mse:X, mae:Y, ms/sample:Z' line it prints.
    Returns (mse, mae, ms_per_sample). seed=None uses run_longExp.py's own
    default (2024). revin=None uses the default (0/off)."""
    model_id = (f'ETTh1_720_{pred_len}'
                + (f'_seed{seed}' if seed is not None else '')
                + (f'_dm{d_model}' if d_model != 512 else '')
                + (f'_revin{revin}' if revin is not None else ''))
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', 'ETTh1',
        '--root_path', './dataset/', '--data_path', 'ETTh1.csv',
        '--features', 'M', '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', '24', '--enc_in', '7', '--d_model', str(d_model),
        '--dropout', '0.1', '--rnn_type', 'gru', '--dec_way', 'pmf', '--channel_id', '1',
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', '64', '--learning_rate', '0.0003',
    ]
    if mark_dim is not None:
        cmd += ['--mark_dim', str(mark_dim)]
    if hour_emb_dim is not None:
        cmd += ['--hour_emb_dim', str(hour_emb_dim)]
    if weekday_emb_dim is not None:
        cmd += ['--weekday_emb_dim', str(weekday_emb_dim)]
    if seed is not None:
        cmd += ['--random_seed', str(seed)]
    if revin is not None:
        cmd += ['--revin', str(revin)]
    if power_transform is not None:
        cmd += ['--power_transform', str(power_transform)]
    if save_preds is not None:
        cmd += ['--save_preds', str(save_preds)]
    if rocket_kernels is not None:
        cmd += ['--rocket_kernels', str(rocket_kernels)]
    if rocket_kernel_size is not None:
        cmd += ['--rocket_kernel_size', str(rocket_kernel_size)]

    print(f'\n{"="*70}\n{model}  H={pred_len}  d_model={d_model}'
          + (f'  seed={seed}' if seed is not None else '')
          + (f'  revin={revin}' if revin is not None else '') + f'\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{model} H={pred_len} failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+), ms/sample:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae/ms-per-sample in output for {model} H={pred_len}')
    return float(m.group(1)), float(m.group(2)), float(m.group(3))


# dataviz-validated categorical palette, fixed order
COLORS = {
    'Paper': '#2a78d6', 'Reconstruction': '#008300',
    'Naive': '#e87ba4', 'Seasonal-naive': '#eda100', 'Improved': '#1baf7a',
    'RevIN': '#4a3aa7', 'Attention': '#e34948',
    'Yeo-Johnson': '#0a8fa3', 'ROCKET': '#a35a00', 'Ensemble (3-seed)': '#9c2c8f',
}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, save_path=None):
    """series: list of (label, {horizon: value}), in display order.
    Always creates a brand-new figure."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(f'SegRNN on ETTh1 — {metric_name.upper()}', color=INK_PRIMARY, fontsize=13, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.14),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')

In [ ]:
from scripts.baselines import build_dataset, windowed_forecasts, seasonal_naive_scale, mase as mase_fn, infer_period
from utils.metrics import MSE, MAE
import argparse as _argparse

def run_baseline_horizon(pred_len):
    """Naive + seasonal-naive on the exact same Dataset/split/scaling as
    run_longExp.py (reuses scripts/baselines.py directly, no duplication)."""
    args = _argparse.Namespace(
        data='ETTh1', root_path='./dataset/', data_path='ETTh1.csv',
        seq_len=720, pred_len=pred_len, features='M', target='OT', freq='h',
        period=None,
    )
    period = infer_period(args)
    train_ds = build_dataset(args, 'train')
    test_ds = build_dataset(args, 'test')
    scale = seasonal_naive_scale(train_ds.data_x, period)
    trues, naive_preds, seasonal_preds = windowed_forecasts(test_ds.data_x, args.seq_len, args.pred_len, period)

    out = {}
    for name, preds in [('naive', naive_preds), ('seasonal_naive', seasonal_preds)]:
        out[name] = {'mse': MSE(preds, trues), 'mae': MAE(preds, trues), 'mase': mase_fn(preds, trues, scale)}
    return out

## Parts 0a–0d — four new Stage 2 strands (Yeo-Johnson, AIC/BIC, ensembling, ROCKET)

Placed immediately after setup, ahead of Parts 1–9, so they can be run on a
fresh runtime without re-running the earlier (already-completed) strands
first. Each is tested independently against the same reconstruction
baseline (`RECON_BASELINE`, defined above — hardcoded from
`results/runs.csv`, the exact numbers Parts 1/4/5/8/9 also compare
against), not stacked with each other or with the calendar-feature/RevIN/
Attention strands — the same reason those four were kept independent
(`docs/stage2_revin_attn_draft.md`'s Discussion): stacking an untested idea
onto an already-negative strand would make it impossible to attribute the
result to either one.

## Part 0a — Yeo-Johnson power transform (preprocessing)

Swaps `StandardScaler` for a per-channel Yeo-Johnson power transform
(`sklearn.preprocessing.PowerTransformer`, `standardize=True`) in
`data_provider/data_loader.py`'s `Dataset_ETT_hour` — fit on train only,
disabled by default (`--power_transform 0` reproduces the plain
reconstruction exactly). `docs/Pre-precessing.pdf`'s preprocessing slide
lists Yeo-Johnson as the standard fix for heteroscedasticity/non-normality
(unlike Box-Cox, it handles negative values, which ETTh1's load/OT
channels contain). All four horizons.

In [ ]:
yeojohnson_results = {}
for h in HORIZONS:
    yeojohnson_results[h] = run_horizon('SegRNN', h, power_transform=1)

yj_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Yeo-Johnson', {h: yeojohnson_results[h][0] for h in HORIZONS}),
]
yj_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Yeo-Johnson', {h: yeojohnson_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', yj_series_mse))
display(make_table('mae', yj_series_mae))
plot_metric('mse', yj_series_mse, save_path='results/figures/yeojohnson_comparison_mse.png')
plot_metric('mae', yj_series_mae, save_path='results/figures/yeojohnson_comparison_mae.png')

## Part 0b — AIC/BIC post-hoc analysis of the `d_model` sweep

Free: zero new training. `docs/Time-Series Forecasting.pdf`'s model-
selection slide gives AIC = n·ln(MSE) + 2k and BIC = n·ln(MSE) + k·ln(n)
(Gaussian-error regression form) as the formal version of "prefer the
simplest model that explains the data well" — the same idea behind the
Theta-method quote from the same deck ("simple methods often outperform
complex ones"). Applies this directly to Part 7's already-completed sweep
(`results/runs.csv`): `n` = test windows × horizon × channels for that
split, `k` = SegRNN's exact analytical parameter count
(`scripts/plot_efficiency_sweep.py`'s formula, GRU-dominated). Reads
`results/runs.csv` and `dataset/ETTh1.csv` only — no GPU, no training.

In [ ]:
SEG_LEN_AB, ENC_IN_AB = 24, 7
D_MODELS_AB = [512, 256, 128, 64]

def param_count(d_model, seg_num_y):
    """Exact SegRNN parameter count (channel_id=1, revin=0) -- matches
    scripts/plot_efficiency_sweep.py."""
    value_embedding = d_model * (SEG_LEN_AB + 1)
    gru = 6 * d_model ** 2 + 6 * d_model
    pos_emb = seg_num_y * (d_model // 2)
    channel_emb = ENC_IN_AB * (d_model // 2)
    predict = SEG_LEN_AB * (d_model + 1)
    return value_embedding + gru + pos_emb + channel_emb + predict

runs_df = pd.read_csv('results/runs.csv')
# exact flags string the Part 7 sweep logged -- avoids picking up unrelated
# SegRNN rows (e.g. Yeo-Johnson/RevIN reruns) that also happen to use d_model=512.
# NOTE: use runs_df['flags'], not runs_df.flags -- DataFrame.flags is a real
# pandas attribute (added 1.2+) and shadows a column literally named "flags".
sweep = runs_df[(runs_df.model == 'SegRNN') &
                 (runs_df['flags'].isin([f'seg_len=24;d_model={d}' for d in D_MODELS_AB])) &
                 (runs_df.seed == 2024)]

aicbic_rows = []
for h in HORIZONS:
    ab_args = _argparse.Namespace(data='ETTh1', root_path='./dataset/', data_path='ETTh1.csv',
                                   seq_len=720, pred_len=h, features='M', target='OT', freq='h', period=None)
    n_windows = len(build_dataset(ab_args, 'test'))
    n = n_windows * h * ENC_IN_AB
    for d in D_MODELS_AB:
        row = sweep[(sweep.horizon == h) & (sweep.d_model == d)]
        mse = float(row.mse.iloc[0])
        k = param_count(d, h // SEG_LEN_AB)
        aic = n * np.log(mse) + 2 * k
        bic = n * np.log(mse) + k * np.log(n)
        aicbic_rows.append({'Horizon': h, 'd_model': d, 'MSE': mse, 'params': k, 'n': n,
                             'AIC': round(aic), 'BIC': round(bic)})

aicbic_df = pd.DataFrame(aicbic_rows)
display(aicbic_df.set_index(['Horizon', 'd_model']))

print('Which d_model each criterion prefers, per horizon:')
for h in HORIZONS:
    sub = aicbic_df[aicbic_df.Horizon == h]
    best_mse = sub.loc[sub.MSE.idxmin(), 'd_model']
    best_aic = sub.loc[sub.AIC.idxmin(), 'd_model']
    best_bic = sub.loc[sub.BIC.idxmin(), 'd_model']
    print(f'  H={h}: MSE picks d_model={best_mse}, AIC picks d_model={best_aic}, BIC picks d_model={best_bic}')

## Part 0c — Multi-seed prediction ensembling

Not lecture-grounded (unlike 0a/0b/0d) — flagged honestly as a general
technique, not one from `docs/Pre-precessing.pdf` or
`docs/Time-Series Forecasting.pdf`. Different in kind from Part 6's seed-
variance check: Part 6 reruns the same config at 3 seeds and reports
*mean ± std of the metrics*; this reruns the same config at 3 seeds, saves
raw predictions (`--save_preds 1`, wired into `exp/exp_main.py`'s `test()`),
**averages the predictions themselves**, and scores the averaged
prediction — a real ensemble, not just a variance estimate. All 3 seeds
must be rerun with `--save_preds 1` (Part 1's original seed=2024 run didn't
save predictions), so there's no reuse from Parts 1/6 here. Scoped to 2
horizons (336, 720 — same choice as Part 6, same compute-budget reason):
6 new runs instead of 12.

In [ ]:
import glob
from utils.metrics import MSE, MAE

ENSEMBLE_SEEDS = [2021, 2022, 2024]
ENSEMBLE_HORIZONS = [336, 720]

def load_preds(h, seed):
    model_id = f'ETTh1_720_{h}_seed{seed}'
    matches = glob.glob(f'results/{model_id}_SegRNN_*/pred.npy')
    if not matches:
        raise FileNotFoundError(f'no pred.npy for H={h} seed={seed} -- rerun with save_preds=1')
    folder = os.path.dirname(matches[0])
    return np.load(os.path.join(folder, 'pred.npy')), np.load(os.path.join(folder, 'true.npy'))

ensemble_single_seed = {}  # per-seed metrics, for the mean-of-metrics comparison
ensemble_results = {}      # averaged-prediction ensemble metrics
for h in ENSEMBLE_HORIZONS:
    per_seed_preds = []
    trues_ref = None
    for seed in ENSEMBLE_SEEDS:
        mse, mae, _ = run_horizon('SegRNN', h, seed=seed, save_preds=1)
        ensemble_single_seed[(h, seed)] = (mse, mae)
        preds, trues = load_preds(h, seed)
        per_seed_preds.append(preds)
        trues_ref = trues if trues_ref is None else trues_ref
    avg_pred = np.mean(per_seed_preds, axis=0)
    ensemble_results[h] = (MSE(avg_pred, trues_ref), MAE(avg_pred, trues_ref))

print('Per-seed metrics (mean of metrics, like Part 6):')
for h in ENSEMBLE_HORIZONS:
    mses = [ensemble_single_seed[(h, s)][0] for s in ENSEMBLE_SEEDS]
    print(f'  H={h}: seed MSEs={[round(m,4) for m in mses]}, mean={statistics.mean(mses):.4f}')

print()
print('Ensembled (predictions averaged, then scored):')
rows = []
for h in ENSEMBLE_HORIZONS:
    ens_mse, ens_mae = ensemble_results[h]
    print(f"  H={h}: MSE={ens_mse:.4f}, MAE={ens_mae:.4f}  "
          f"(vs. Reconstruction MSE={RECON_BASELINE['mse'][h]:.4f}, MAE={RECON_BASELINE['mae'][h]:.4f})")
    rows.append({'Horizon': h, 'Reconstruction MSE': RECON_BASELINE['mse'][h], 'Ensemble MSE': round(ens_mse, 4),
                 'Reconstruction MAE': RECON_BASELINE['mae'][h], 'Ensemble MAE': round(ens_mae, 4)})
display(pd.DataFrame(rows).set_index('Horizon'))

## Part 0d — ROCKET-style random-convolutional features (`SegRNNRocket`)

The most speculative of the four new strands (approved anyway) — ROCKET
(`docs/Pre-precessing.pdf`'s feature-engineering slide: random
convolutional kernels, Max + PPV pooling) is a classification-era
technique; there's no textbook precedent for using it inside a
forecaster, so this is an honest adaptation, not a documented method (see
`models/SegRNNRocket.py`'s docstring). Deliberately repeats the same
"inject extra information into `h_n`" pattern that all three calendar-
feature attempts used and that regressed every time — but with a
completely different, non-calendar information source (fixed random
convolutions of the raw window itself), as an independent check of
whether that earlier finding is calendar-specific or a more general
property of this architecture/dataset. All four horizons.

In [ ]:
rocket_results = {}
for h in HORIZONS:
    rocket_results[h] = run_horizon('SegRNNRocket', h)

rocket_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('ROCKET', {h: rocket_results[h][0] for h in HORIZONS}),
]
rocket_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('ROCKET', {h: rocket_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', rocket_series_mse))
display(make_table('mae', rocket_series_mae))
plot_metric('mse', rocket_series_mse, save_path='results/figures/rocket_comparison_mse.png')
plot_metric('mae', rocket_series_mae, save_path='results/figures/rocket_comparison_mae.png')

## Part 1 — Reconstruction (SegRNN)

Trains the paper's own reference configuration
(`scripts/SegRNN/etth1.sh`'s hyperparameters: `seq_len=720`, `seg_len=24`,
`d_model=512`, GRU, PMF decode, channel id on) for all four horizons.

In [ ]:
recon = {}
for h in HORIZONS:
    recon[h] = run_horizon('SegRNN', h)
print('\nReconstruction (SegRNN) done:', recon)

## Part 2 — Classical baselines (naive, seasonal-naive)

Deterministic, no GPU, seconds to run.

In [ ]:
baselines = {}
for h in HORIZONS:
    baselines[h] = run_baseline_horizon(h)
print('Baselines done:', baselines)

## Part 3 — Stage 1 results: paper vs. reconstruction vs. baselines

In [ ]:
def series_for(metric_name, include_improved=False):
    s = [
        ('Paper', PAPER[metric_name]),
        ('Reconstruction', {h: recon[h][0 if metric_name == 'mse' else 1] for h in HORIZONS}),
        ('Naive', {h: baselines[h]['naive'][metric_name] for h in HORIZONS}),
        ('Seasonal-naive', {h: baselines[h]['seasonal_naive'][metric_name] for h in HORIZONS}),
    ]
    if include_improved:
        s.append(('Improved', {h: improved[h][0 if metric_name == 'mse' else 1] for h in HORIZONS}))
    return s

mse_series = series_for('mse')
mae_series = series_for('mae')

display(make_table('mse', mse_series))
display(make_table('mae', mae_series))

plot_metric('mse', mse_series, save_path='results/figures/mse_comparison.png')
plot_metric('mae', mae_series, save_path='results/figures/mae_comparison.png')

## Part 4 — Improved architecture (SegRNNTime, attempt 4: decoder-side)

Three earlier attempts (see `docs/stage2_report_draft.md`) injected
calendar features into the **encoder** and all consistently hurt
MSE/MAE — likely because the encoder already compresses the whole
look-back window into a single hidden state `h_n` (an "information
bottleneck", per `docs/DL_for_TS.pdf`'s Attention slide) before the
decoder ever sees it, so calendar info had to compete with raw values for
space in that one vector.

This attempt moves the calendar signal to the **decoder** instead, via
the PMF decoder's positional embedding `PE = concat(rp, cp)` — which the
paper's own ablation shows is the single highest-leverage component in
the architecture, and which bypasses the encoder bottleneck entirely.
`PE` is extended to `concat(rp, cp, hour_embedding, weekday_embedding)`:
learned embeddings (24 and 7 categories, `docs/DL_for_TS.pdf`'s
"embed the cyclical feature" pattern) of each target segment's own
**future** hour-of-day and day-of-week — known in advance for the whole
forecast horizon, not leakage. The encoder goes back to being identical
to plain `SegRNN.py`. Same hyperparameters, split, and seed as Part 1.

In [ ]:
improved = {}
for h in HORIZONS:
    improved[h] = run_horizon('SegRNNTime', h, mark_dim=4, hour_emb_dim=16, weekday_emb_dim=8)
print('\nImproved (SegRNNTime) done:', improved)

## Part 5 — Final results: paper vs. reconstruction vs. baselines vs. improved

In [ ]:
mse_series_final = series_for('mse', include_improved=True)
mae_series_final = series_for('mae', include_improved=True)

display(make_table('mse', mse_series_final))
display(make_table('mae', mae_series_final))

plot_metric('mse', mse_series_final, save_path='results/figures/mse_comparison_final.png')
plot_metric('mae', mae_series_final, save_path='results/figures/mae_comparison_final.png')

## Part 6 — Seed variance check

Everything above uses a single seed (2024) per configuration. Before
reading anything into the small deltas in Part 5, this checks whether
they're a real effect or just seed-to-seed noise: reruns Reconstruction
and Improved at **2 more seeds** (2021, 2022) for the two most relevant
horizons (336 — smallest observed gap; 720 — largest), combined with the
seed=2024 results already computed in Parts 1/4 (no need to rerun those).
3 seeds × 2 horizons × 2 models = 8 new training runs.

In [ ]:
SEED_TEST_SEEDS = [2021, 2022, 2024]  # 2024 reuses Part 1/4 results, no rerun needed
SEED_TEST_HORIZONS = [336, 720]

seed_results = {'SegRNN': {}, 'SegRNNTime': {}}
for h in SEED_TEST_HORIZONS:
    seed_results['SegRNN'][(h, 2024)] = recon[h]
    seed_results['SegRNNTime'][(h, 2024)] = improved[h]

for seed in [2021, 2022]:
    for h in SEED_TEST_HORIZONS:
        seed_results['SegRNN'][(h, seed)] = run_horizon('SegRNN', h, seed=seed)
        seed_results['SegRNNTime'][(h, seed)] = run_horizon(
            'SegRNNTime', h, seed=seed, mark_dim=4, hour_emb_dim=16, weekday_emb_dim=8)

print('\nSeed test done:', seed_results)

In [ ]:
def seed_stats(model, h, metric_idx):
    vals = [seed_results[model][(h, s)][metric_idx] for s in SEED_TEST_SEEDS]
    mean = statistics.mean(vals)
    std = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return vals, mean, std

summary_rows = []
for h in SEED_TEST_HORIZONS:
    for model, label in [('SegRNN', 'Reconstruction'), ('SegRNNTime', 'Improved')]:
        _, mse_mean, mse_std = seed_stats(model, h, 0)
        _, mae_mean, mae_std = seed_stats(model, h, 1)
        summary_rows.append({
            'Horizon': h, 'Model': label,
            'MSE mean': round(mse_mean, 4), 'MSE std': round(mse_std, 4),
            'MAE mean': round(mae_mean, 4), 'MAE std': round(mae_std, 4),
        })
display(pd.DataFrame(summary_rows).set_index(['Horizon', 'Model']))

print('Paired per-seed delta (Improved - Reconstruction), MSE:')
for h in SEED_TEST_HORIZONS:
    deltas = [seed_results['SegRNNTime'][(h, s)][0] - seed_results['SegRNN'][(h, s)][0] for s in SEED_TEST_SEEDS]
    mean_delta = statistics.mean(deltas)
    std_delta = statistics.stdev(deltas) if len(deltas) > 1 else float('nan')
    verdict = 'likely NOISE (|mean| < std)' if abs(mean_delta) < std_delta else 'likely a REAL effect (|mean| > std)'
    print(f'  H={h}: per-seed deltas={[round(d,4) for d in deltas]}, mean={mean_delta:+.4f}, std={std_delta:.4f} -> {verdict}')

# error-bar chart, mean +/- std across the 3 seeds, one panel per horizon
fig, axes = plt.subplots(1, len(SEED_TEST_HORIZONS), figsize=(5 * len(SEED_TEST_HORIZONS), 5), facecolor=SURFACE)
for ax, h in zip(axes, SEED_TEST_HORIZONS):
    ax.set_facecolor(SURFACE)
    labels = ['Reconstruction', 'Improved']
    means, stds = [], []
    for model in ['SegRNN', 'SegRNNTime']:
        _, mean, std = seed_stats(model, h, 0)
        means.append(mean)
        stds.append(std)
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=6, color=[COLORS['Reconstruction'], COLORS['Improved']],
                   edgecolor=SURFACE, linewidth=0.5, error_kw={'ecolor': INK_SECONDARY, 'elinewidth': 1.5})
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{mean:.3f}',
                 ha='center', va='bottom', fontsize=9, color=INK_PRIMARY)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, color=INK_SECONDARY)
    ax.set_ylabel('MSE', color=INK_SECONDARY)
    ax.set_title(f'H={h} (mean ± std, n=3 seeds)', color=INK_PRIMARY, fontsize=11, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
fig.suptitle('Seed variance: Reconstruction vs. Improved', color=INK_PRIMARY, fontsize=13, x=0.02, ha='left')
fig.tight_layout()
fig.savefig('results/figures/seed_variance.png', dpi=200, facecolor=SURFACE)
print('saved results/figures/seed_variance.png')
plt.show()

## Part 7 — Efficiency sweep: hidden size (`d_model`)

A second, independent Stage 2 improvement candidate, unrelated to the
calendar-feature work above: **100% unmodified code** — `--d_model` is
already a flag on the original `run_longExp.py`/`models/SegRNN.py`, so
this is purely a hyperparameter sweep, not an architecture change.
Reruns the reconstruction at `d_model ∈ {512, 256, 128, 64}` across all
four horizons (`d_model=512` reruns fresh here too, so `ms/sample` is
captured consistently with the other three) to trace out an
accuracy-vs-inference-cost curve. Directly matches the assignment's
"improve computational efficiency" example. 4 values × 4 horizons = 16
runs.

In [ ]:
D_MODEL_VALUES = [512, 256, 128, 64]

sweep_results = {}
for d in D_MODEL_VALUES:
    for h in HORIZONS:
        sweep_results[(d, h)] = run_horizon('SegRNN', h, d_model=d)

print('\nEfficiency sweep done:', sweep_results)

In [ ]:
sweep_rows = []
for d in D_MODEL_VALUES:
    for h in HORIZONS:
        mse, mae, ms = sweep_results[(d, h)]
        sweep_rows.append({'d_model': d, 'Horizon': h, 'MSE': round(mse, 4), 'MAE': round(mae, 4),
                            'ms/sample': round(ms, 4)})
sweep_df = pd.DataFrame(sweep_rows)

print('MSE by horizon x d_model')
display(sweep_df.pivot(index='Horizon', columns='d_model', values='MSE'))
print('\nInference cost (ms/sample) by horizon x d_model')
display(sweep_df.pivot(index='Horizon', columns='d_model', values='ms/sample'))

# accuracy-vs-efficiency curves: one line per horizon, x=d_model (log2 scale
# since values are powers of 2), left panel MSE, right panel inference cost.
# dataviz palette, first 4 slots (validated for all-pairs comparisons)
horizon_colors = {96: '#2a78d6', 192: '#008300', 336: '#e87ba4', 720: '#eda100'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=SURFACE)
for ax, metric, ylabel, title in [
    (ax1, 'MSE', 'MSE', 'Accuracy vs. hidden size'),
    (ax2, 'ms/sample', 'Inference time (ms/sample)', 'Inference cost vs. hidden size'),
]:
    ax.set_facecolor(SURFACE)
    for h in HORIZONS:
        sub = sweep_df[sweep_df['Horizon'] == h].sort_values('d_model')
        ax.plot(sub['d_model'], sub[metric], marker='o', color=horizon_colors[h],
                 label=f'H={h}', linewidth=2, markersize=6)
    ax.set_xscale('log', base=2)
    ax.set_xticks(D_MODEL_VALUES)
    ax.set_xticklabels([str(d) for d in D_MODEL_VALUES], color=INK_SECONDARY)
    ax.set_xlabel('d_model', color=INK_SECONDARY)
    ax.set_ylabel(ylabel, color=INK_SECONDARY)
    ax.set_title(title, color=INK_PRIMARY, fontsize=12, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(BASELINE_AXIS)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK_SECONDARY)

fig.tight_layout()
fig.savefig('results/figures/efficiency_sweep.png', dpi=200, facecolor=SURFACE)
print('saved results/figures/efficiency_sweep.png')
plt.show()

## Part 8 — RevIN normalization

A third, independent Stage 2 improvement candidate. `layers/RevIN.py` and
its wiring into `models/SegRNN.py` already exist in the repo, unused —
`--revin` defaults to `0`. RevIN replaces SegRNN's default normalization
(subtract the input window's *last value*, add it back after decoding)
with per-window z-score standardization (`z=(x-μ)/σ`, then
`x̂=zσ+μ` — the exact formula from `docs/DL_for_TS.pdf`'s "Embedding
Layer" slide) computed over the *whole* look-back window rather than just
its last point. Reruns the reconstruction with `--revin 1`, all four
horizons — genuinely zero new code, just a flag.

In [ ]:
revin_results = {}
for h in HORIZONS:
    revin_results[h] = run_horizon('SegRNN', h, revin=1)

revin_series_mse = [
    ('Reconstruction', {h: recon[h][0] for h in HORIZONS}),
    ('RevIN', {h: revin_results[h][0] for h in HORIZONS}),
]
revin_series_mae = [
    ('Reconstruction', {h: recon[h][1] for h in HORIZONS}),
    ('RevIN', {h: revin_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', revin_series_mse))
display(make_table('mae', revin_series_mae))
plot_metric('mse', revin_series_mse, save_path='results/figures/revin_comparison_mse.png')
plot_metric('mae', revin_series_mae, save_path='results/figures/revin_comparison_mae.png')

## Part 9 — Attention over encoder states (`SegRNNAttn`)

A fourth, independent Stage 2 improvement candidate — and the most
structural one: rather than adding *more* information for the encoder's
single bottleneck vector `h_n` to carry (what all four calendar-feature
attempts did), this removes the bottleneck itself. `models/SegRNNAttn.py`
keeps the encoder's full per-segment output sequence (all `n=30` states,
not just the final `h_n`) and lets the PMF decoder attend over all of
them at each decode step — standard scaled dot-product attention, the
textbook fix for the "information bottleneck" `docs/DL_for_TS.pdf`'s
Attention slide names directly. The shared-GRU-cell trick central to
SegRNN's design (same cell for encode and decode) is preserved — only
the decode step's *input* changes (positional embedding + attention
context, instead of positional embedding alone). No new CLI flags
needed. Same hyperparameters, split, and seed as Part 1.

In [ ]:
attn_results = {}
for h in HORIZONS:
    attn_results[h] = run_horizon('SegRNNAttn', h)

attn_series_mse = [
    ('Reconstruction', {h: recon[h][0] for h in HORIZONS}),
    ('Attention', {h: attn_results[h][0] for h in HORIZONS}),
]
attn_series_mae = [
    ('Reconstruction', {h: recon[h][1] for h in HORIZONS}),
    ('Attention', {h: attn_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', attn_series_mse))
display(make_table('mae', attn_series_mae))
plot_metric('mse', attn_series_mse, save_path='results/figures/attn_comparison_mse.png')
plot_metric('mae', attn_series_mae, save_path='results/figures/attn_comparison_mae.png')

## Optional — save results back into the repo

Writes `results/runs.csv` fresh from everything computed above and stages
it plus the figures. Commit/push are left commented out on purpose --
review `git status`/`git diff` first, then uncomment when you're ready.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []

for h in HORIZONS:
    mse, mae, ms = recon[h]
    rows.append([f'SegRNN_ETTh1_{h}_{ts}', ts, 'SegRNN', 'ETTh1', h, 720, 24, 512, 2024,
                 'seg_len=24;d_model=512', mse, mae, '', '', '', '', 'reconstruction'])

for h in HORIZONS:
    for name in ['naive', 'seasonal_naive']:
        b = baselines[h][name]
        rows.append([f'{name}_ETTh1_{h}_{ts}', ts, name, 'ETTh1', h, 720, '', '', 2024,
                     'period=24;features=M;seq_len=720', b['mse'], b['mae'], b['mase'],
                     '', '', '', 'deterministic baseline, no training'])

for h in HORIZONS:
    mse, mae, ms = improved[h]
    rows.append([f'SegRNNTime_ETTh1_{h}_{ts}', ts, 'SegRNNTime', 'ETTh1', h, 720, 24, 512, 2024,
                 'seg_len=24;d_model=512;mark_dim=4;hour_emb_dim=16;weekday_emb_dim=8', mse, mae, '', '', '', '',
                 'improved: decoder-side hour+weekday embedding in PE'])

if 'sweep_results' in dir():
    for (d, h), (mse, mae, ms) in sweep_results.items():
        rows.append([f'SegRNN_ETTh1_{h}_dm{d}_{ts}', ts, 'SegRNN', 'ETTh1', h, 720, 24, d, 2024,
                     f'seg_len=24;d_model={d}', mse, mae, '', '', '', '', f'efficiency sweep: d_model={d}'])

if 'revin_results' in dir():
    for h, (mse, mae, ms) in revin_results.items():
        rows.append([f'SegRNN_ETTh1_{h}_revin1_{ts}', ts, 'SegRNN', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512;revin=1', mse, mae, '', '', '', '', 'revin normalization'])

if 'attn_results' in dir():
    for h, (mse, mae, ms) in attn_results.items():
        rows.append([f'SegRNNAttn_ETTh1_{h}_{ts}', ts, 'SegRNNAttn', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512', mse, mae, '', '', '', '', 'attention over encoder states'])

with open('results/runs.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(RUNS_CSV_HEADER)
    writer.writerows(rows)

print(f'wrote {len(rows)} rows to results/runs.csv')

!git add results/runs.csv results/figures/
!git status
# review the diff above, then when ready:
# !git commit -m "Update results: reconstruction, baselines, improved model"
# !git push origin main